# V. Xuất data - Import Power BI

In [ ]:
# VERIFY 
import pandas as pd

checks = {
    "df_yoy":       ["Year","HS2","Total_Export_Value","YoY_Growth_%"],
    "total_by_year":["Year","Total_Export","YoY_Growth_%"],
    "group_by_year":["Year","Industry_Group","Group_Export","Export_Share_%"],
    "cr_df":        ["Year","CR5_%","CR10_%","HHI"],
    "quadrant_df":  ["Industry_Group","Avg_Export_USD","YoY_Std","Quadrant"],
    "kpi_summary":  ["Industry_Group","ESI","Risk_Score","Risk_Level"],
    "hs_list":      ["HS2","HS_Desc"],
    "dfA":          ["Year","HS2","Total_Export_Value","Industry_Group"],
}

print("KIỂM TRA BIẾN GỐC:")
print("=" * 60)
all_ok = True
for varname, expected_cols in checks.items():
    try:
        df_check = eval(varname)
        missing = [c for c in expected_cols if c not in df_check.columns]
        if missing:
            print(f"⚠️  {varname:20s} {str(df_check.shape):12s} THIẾU CỘT: {missing}")
            all_ok = False
        else:
            print(f" {varname:20s} {str(df_check.shape):12s} OK")
    except NameError:
        print(f"❌ {varname:20s} KHÔNG TỒN TẠI — cần chạy lại notebook")
        all_ok = False

print("=" * 60)
if all_ok:
    print(" TẤT CẢ OK — có thể chạy cell export an toàn")
else:
    print("CÓ VẤN ĐỀ — cần fix trước khi export")

In [ ]:
# ============================================================
# EXPORT DATA FOR POWER BI — FINAL VERSION
# 4 FACT TABLES + 5 DIMENSION TABLES + 1 HELPER TABLE
# - fact_group_by_year đã được gộp vào fact_export
# - dim_industry được giữ lại theo đúng Star Schema đã vẽ
# ============================================================

import os
import re
import pandas as pd

# ------------------------------------------------------------
# 0. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = {
    "df_yoy": ["Year", "HS2", "Total_Export_Value", "YoY_Growth_%"],
    "dfA": ["Year", "HS2", "Total_Export_Value", "Industry_Group"],
    "hs_list": ["HS2", "HS_Desc"],
    "total_by_year": ["Year", "Total_Export", "YoY_Growth_%"],
    "cr_df": ["Year", "CR5_%", "CR10_%", "HHI"],
    "quadrant_df": ["Industry_Group", "Avg_Export_USD", "YoY_Std", "Quadrant"],
    "kpi_summary": [
        "Industry_Group", "Avg_Export_USD", "Avg_Share_pct",
        "ESI", "YoY_Mean", "YoY_Std", "Recovery_Rate",
        "Recovery_Label", "Risk_Score", "Risk_Level"
    ],
}

print("=" * 70)
print("KIỂM TRA BIẾN ĐẦU VÀO")
print("=" * 70)

all_ok = True

for var_name, cols in required_objects.items():
    if var_name not in globals():
        print(f"❌ {var_name}: chưa tồn tại. Cần chạy lại các cell phía trên.")
        all_ok = False
        continue

    df_check = globals()[var_name]
    missing_cols = [c for c in cols if c not in df_check.columns]

    if missing_cols:
        print(f"⚠️  {var_name}: thiếu cột {missing_cols}")
        all_ok = False
    else:
        print(f"✅ {var_name:22s} {str(df_check.shape):14s} OK")

if not all_ok:
    raise ValueError("Một số biến/cột đầu vào chưa đủ. Vui lòng kiểm tra lại các cell phía trên.")

print("=" * 70)
print("TẤT CẢ BIẾN ĐẦU VÀO ĐÃ SẴN SÀNG")
print("=" * 70)


# ------------------------------------------------------------
# 1. DIM YEAR
# ------------------------------------------------------------

dim_year = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023],
    "Year_Label": ["2019", "2020", "2021", "2022", "2023"],
    "Period": ["Pre-COVID", "COVID", "Recovery", "Peak", "Adjustment"],
    "Year_Order": [1, 2, 3, 4, 5],
})


# ------------------------------------------------------------
# 2. DIM INDUSTRY
# Bảng chiều ở cấp HS2: HS2, mô tả hàng hóa, nhóm ngành
# Nối với fact_export qua khóa HS2
# ------------------------------------------------------------

dim_industry = (
    hs_list[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .copy()
)

dim_industry["HS2"] = dim_industry["HS2"].astype(str).str.zfill(2)

hs_to_group = (
    dfA[["HS2", "Industry_Group"]]
    .drop_duplicates()
    .copy()
)

hs_to_group["HS2"] = hs_to_group["HS2"].astype(str).str.zfill(2)

dim_industry = (
    dim_industry
    .merge(hs_to_group, on="HS2", how="left")
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. DIM INDUSTRY GROUP
# Bảng chiều ở cấp nhóm ngành, dùng cho các fact tổng hợp
# Nối với fact_volatility_quadrant và fact_risk_summary
# ------------------------------------------------------------

dim_industry_group = (
    dim_industry[["Industry_Group"]]
    .drop_duplicates()
    .dropna()
    .sort_values("Industry_Group")
    .reset_index(drop=True)
)

dim_industry_group["Industry_Group_ID"] = range(1, len(dim_industry_group) + 1)


# ------------------------------------------------------------
# 4. DIM QUADRANT
# Bảng chiều phân loại quy mô - biến động
# ------------------------------------------------------------

def extract_quadrant_order(q):
    match = re.search(r"Q(\d)", str(q))
    return int(match.group(1)) if match else None

def build_quadrant_desc(q):
    q_text = str(q)

    if "Q1" in q_text:
        return "Quy mô lớn - biến động thấp"
    if "Q2" in q_text:
        return "Quy mô lớn - biến động cao"
    if "Q3" in q_text:
        return "Quy mô nhỏ - biến động thấp"
    if "Q4" in q_text:
        return "Quy mô nhỏ - biến động cao"

    return "Nhóm phân loại quy mô - biến động"

dim_quadrant = (
    quadrant_df[["Quadrant"]]
    .drop_duplicates()
    .dropna()
    .copy()
)

dim_quadrant["Quadrant_Order"] = dim_quadrant["Quadrant"].apply(extract_quadrant_order)
dim_quadrant["Quadrant_Desc"] = dim_quadrant["Quadrant"].apply(build_quadrant_desc)

dim_quadrant = (
    dim_quadrant
    .sort_values("Quadrant_Order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. DIM RISK LEVEL
# Bảng chiều mức rủi ro, giúp Power BI sort đúng thứ tự
# ------------------------------------------------------------

dim_risk_level = pd.DataFrame({
    "Risk_Level": ["Thấp", "Trung bình", "Cao"],
    "Risk_Order": [1, 2, 3],
    "Risk_Desc": [
        "Rủi ro tương đối thấp",
        "Rủi ro cần theo dõi",
        "Rủi ro cần xem xét thận trọng"
    ]
})


# ------------------------------------------------------------
# 6. FACT EXPORT
# Bảng fact chi tiết Year - HS2
# Dùng chung cho Dashboard 1 và Dashboard 2
# ------------------------------------------------------------

fact_export = df_yoy[[
    "Year", "HS2", "Total_Export_Value", "YoY_Growth_%"
]].copy()

fact_export["HS2"] = fact_export["HS2"].astype(str).str.zfill(2)

# Bổ sung Industry_Group để có thể tổng hợp trực tiếp theo nhóm ngành
fact_export = fact_export.merge(
    dim_industry[["HS2", "Industry_Group"]],
    on="HS2",
    how="left"
)

# Đổi USD sang triệu USD
fact_export["Total_Export_Value_M"] = (
    fact_export["Total_Export_Value"] / 1_000_000
).round(2)

# Kiểm tra YoY outlier trước khi cap
bi_cap = fact_export[
    (fact_export["YoY_Growth_%"] > 500) |
    (fact_export["YoY_Growth_%"] < -100)
][["Year", "HS2", "Industry_Group", "YoY_Growth_%"]].dropna().sort_values(
    "YoY_Growth_%", ascending=False
)

print("\nKIỂM TRA YOY OUTLIER")
print("=" * 70)
print(f"Số bản ghi có YoY ngoài [-100%, +500%]: {len(bi_cap)}")

if len(bi_cap) > 0:
    print("Chi tiết các bản ghi bị cap:")
    print(bi_cap.to_string(index=False))
else:
    print("Không có bản ghi nào bị cap.")

# Cap YoY outlier để tránh làm lệch trục biểu đồ Power BI
fact_export["YoY_Growth_%"] = (
    fact_export["YoY_Growth_%"].round(2).clip(-100, 500)
)

# Tính Export Share theo tổng kim ngạch từng năm
total_per_year = fact_export.groupby("Year")["Total_Export_Value_M"].transform("sum")

fact_export["Export_Share_%"] = (
    fact_export["Total_Export_Value_M"] / total_per_year * 100
).round(2)

fact_export = fact_export[[
    "Year",
    "HS2",
    "Industry_Group",
    "Total_Export_Value_M",
    "YoY_Growth_%",
    "Export_Share_%"
]]


# ------------------------------------------------------------
# 7. HELPER: TOTAL BY YEAR
# Bảng hỗ trợ hiển thị KPI tổng quan
# Không xem là fact chính trong Star Schema
# ------------------------------------------------------------

total_by_year_export = total_by_year.copy()

total_by_year_export["Total_Export_B"] = (
    total_by_year_export["Total_Export"] / 1_000_000_000
).round(2)

total_by_year_export["YoY_Growth_%"] = (
    total_by_year_export["YoY_Growth_%"].round(2)
)

total_by_year_export = total_by_year_export[[
    "Year", "Total_Export_B", "YoY_Growth_%"
]]


# ------------------------------------------------------------
# 8. FACT CONCENTRATION
# CR5, CR10, HHI theo năm
# ------------------------------------------------------------

fact_concentration = cr_df.copy()

fact_concentration["CR5_%"] = fact_concentration["CR5_%"].round(2)
fact_concentration["CR10_%"] = fact_concentration["CR10_%"].round(2)
fact_concentration["HHI"] = fact_concentration["HHI"].round(4)

fact_concentration = fact_concentration[[
    "Year", "CR5_%", "CR10_%", "HHI"
]]


# ------------------------------------------------------------
# 9. FACT VOLATILITY QUADRANT
# Avg Export, YoY Std, ESI, Quadrant theo nhóm ngành
# ------------------------------------------------------------

fact_volatility_quadrant = quadrant_df.copy()

# Bổ sung ESI từ kpi_summary để phục vụ Dashboard 4
fact_volatility_quadrant = fact_volatility_quadrant.merge(
    kpi_summary[["Industry_Group", "ESI"]],
    on="Industry_Group",
    how="left"
)

fact_volatility_quadrant["Avg_Export_M"] = (
    fact_volatility_quadrant["Avg_Export_USD"] / 1_000_000
).round(2)

fact_volatility_quadrant["YoY_Std"] = fact_volatility_quadrant["YoY_Std"].round(2)
fact_volatility_quadrant["ESI"] = fact_volatility_quadrant["ESI"].round(4)

fact_volatility_quadrant = fact_volatility_quadrant[[
    "Industry_Group", "Avg_Export_M", "YoY_Std", "ESI", "Quadrant"
]]


# ------------------------------------------------------------
# 10. FACT RISK SUMMARY
# Bộ độ đo tổng hợp rủi ro ngành
# Recovery_Rate giữ nguyên theo code hiện tại: 2022 / 2019
# ------------------------------------------------------------

fact_risk_summary = kpi_summary.copy()

fact_risk_summary["Avg_Export_M"] = (
    fact_risk_summary["Avg_Export_USD"] / 1_000_000
).round(2)

fact_risk_summary["Avg_Share_pct"] = fact_risk_summary["Avg_Share_pct"].round(2)
fact_risk_summary["ESI"] = fact_risk_summary["ESI"].round(4)
fact_risk_summary["YoY_Mean"] = fact_risk_summary["YoY_Mean"].round(2)
fact_risk_summary["YoY_Std"] = fact_risk_summary["YoY_Std"].round(2)
fact_risk_summary["Recovery_Rate"] = fact_risk_summary["Recovery_Rate"].round(3)
fact_risk_summary["Risk_Score"] = fact_risk_summary["Risk_Score"].round(3)

fact_risk_summary = fact_risk_summary[[
    "Industry_Group",
    "Avg_Export_M",
    "Avg_Share_pct",
    "ESI",
    "YoY_Mean",
    "YoY_Std",
    "Recovery_Rate",
    "Recovery_Label",
    "Risk_Score",
    "Risk_Level"
]]


# ------------------------------------------------------------
# 11. EXPORT ALL TABLES
# ------------------------------------------------------------

exports = {
    # Dimension tables
    "dim_year.csv": dim_year,
    "dim_industry.csv": dim_industry,
    "dim_industry_group.csv": dim_industry_group,
    "dim_quadrant.csv": dim_quadrant,
    "dim_risk_level.csv": dim_risk_level,

    # Fact tables
    "fact_export.csv": fact_export,
    "fact_concentration.csv": fact_concentration,
    "fact_volatility_quadrant.csv": fact_volatility_quadrant,
    "fact_risk_summary.csv": fact_risk_summary,

    # Helper table
    "total_by_year.csv": total_by_year_export,
}

for filename, df_export in exports.items():
    df_export.to_csv(filename, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# 12. FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KIỂM TRA FILE XUẤT")
print("=" * 70)

for filename, df_export in exports.items():
    size_kb = os.path.getsize(filename) / 1024
    print(f"{filename:35s} {str(df_export.shape):15s} {size_kb:8.1f} KB")

print("\nKIỂM TRA SỐ LIỆU QUAN TRỌNG")
print("=" * 70)

total_5_years_m = fact_export.groupby("Year")["Total_Export_Value_M"].sum().sum()
export_2023_m = fact_export.loc[
    fact_export["Year"] == 2023, "Total_Export_Value_M"
].sum()

print(f"Total XK 5 năm: {total_5_years_m:,.0f} triệu USD")
print(f"KN 2023: {export_2023_m:,.0f} triệu USD")
print(f"Số mã HS2: {fact_export['HS2'].nunique()}")
print(f"Số nhóm ngành: {dim_industry_group['Industry_Group'].nunique()}")

yoy_2023 = total_by_year_export.loc[
    total_by_year_export["Year"] == 2023, "YoY_Growth_%"
].values[0]

print(f"YoY 2023: {yoy_2023}%")

print("\nKIỂM TRA CẤU TRÚC MÔ HÌNH")
print("=" * 70)
print("Fact tables:")
print("- fact_export.csv")
print("- fact_concentration.csv")
print("- fact_volatility_quadrant.csv")
print("- fact_risk_summary.csv")

print("\nDimension tables:")
print("- dim_year.csv")
print("- dim_industry.csv")
print("- dim_industry_group.csv")
print("- dim_quadrant.csv")
print("- dim_risk_level.csv")

print("\nHelper table:")
print("- total_by_year.csv")

print("=" * 70)
print("XUẤT XONG — SẴN SÀNG IMPORT VÀO POWER BI")
print("=" * 70)